In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("config_catalog_name", "", "config_catalog_name")
dbutils.widgets.text("source_catalog_name", "", "source_catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")
dbutils.widgets.text("source_schema_name", "", "source_schema_name")
dbutils.widgets.text("target_schema_name", "", "target_schema_name")
dbutils.widgets.text("table_name", "", "table_name")

config_catalog_name = dbutils.widgets.get("config_catalog_name")
source_catalog_name = dbutils.widgets.get("source_catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")
source_schema_name = dbutils.widgets.get("source_schema_name")
target_schema_name = dbutils.widgets.get("target_schema_name")
table_name = dbutils.widgets.get("table_name")

print("config_catalog_name: ", config_catalog_name)
print("source_catalog_name: ", source_catalog_name)
print("config_schema_name: ", config_schema_name)
print("source_schema_name: ", source_schema_name)
print("target_schema_name: ", target_schema_name)
print("table_name: ", table_name)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {source_catalog_name}.{target_schema_name}")

In [0]:
query = f"""
SELECT
    m.table_name AS table_name,
    r.rule_name,
    m.column_name AS column,
    r.rule_function AS function,
    m.criticality AS criticality,
    m.arguments AS arguments,
    r.rule_type,
    m.is_active
FROM {config_catalog_name}.{config_schema_name}.dqx_rule_mappings m
JOIN {config_catalog_name}.{config_schema_name}.dqx_rule_definitions r ON m.rule_id = r.rule_id
WHERE lower(m.table_name) = lower('{source_catalog_name}.{source_schema_name}.{table_name}')
"""

dqx_mapped_df = spark.sql(query).filter("is_active=true")
display(dqx_mapped_df)

if dqx_mapped_df.isEmpty():
    raise Exception("No active rules found for the table")

## Apply DQX Checks

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule,DQDatasetRule
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.config import TableChecksStorageConfig
from databricks.labs.dqx.config import InputConfig, OutputConfig
from databricks.labs.dqx.metrics_observer import DQMetricsObserver

In [0]:
# 4. Initialize Engine and Run
observer = DQMetricsObserver(name="my_observation")
ws = WorkspaceClient()
engine = DQEngine(ws, observer=observer)

## config - check by metadata

In [0]:
checks_config = []
for row in dqx_mapped_df.collect():
    import json
    args = {}
    if row['arguments']:
        for k, v in row['arguments'].items():
            try:
                args[k] = json.loads(v)
            except Exception:
                args[k] = v
    check_dict = {
        "criticality": row['criticality'],
        "check": {
            "function": row['function'],
            "arguments": {k: (True if str(v).upper() == 'TRUE' else False if str(v).upper() == 'FALSE' else v) for k, v in args.items()}
        }
    }
    checks_config.append(check_dict)

In [0]:
for i in (checks_config):
    print(i)

In [0]:
# end-to-end quality checking flow
engine.apply_checks_by_metadata_and_save_in_table(
    input_config=InputConfig(f"{source_catalog_name}.{source_schema_name}.{table_name}"),
    checks=checks_config,
    output_config=OutputConfig(f"{source_catalog_name}.{target_schema_name}.{table_name}_output", mode="overwrite", options={"mergeSchema": "true"}),
    metrics_config=OutputConfig(f"{config_catalog_name}.{config_schema_name}.summary_metrics", mode="append", options={"mergeSchema": "true"}),
    quarantine_config=None
)

In [0]:
display(spark.table(f"{source_catalog_name}.{target_schema_name}.{table_name}_output"))
display(spark.table(f"{config_catalog_name}.{config_schema_name}.summary_metrics"))